# King CRAB PMT backgrounds: muons, Ba-133, and Eu-152

This notebook extends the cosmogenic-muon analysis to the complete declared optical-background model. It compares the PMT pulse-height and integrated-charge populations produced by:

1. measured terminal-4 dark, electronic, and ambient-light triggers;
2. bare-sky cosmogenic muons crossing the King CRAB target;
3. a $0.34\,\mu\mathrm{Ci}$ Ba-133 source at the simulated source position;
4. a $0.44\,\mu\mathrm{Ci}$ Eu-152 source at the same position.

Nexus photons are grouped into **fixed 10 ns arrival-time bins**. Each photon independently produces a photoelectron with the DATA-derived 128 nm PDE. Field-off distributions use S1 photons. Field-on distributions retain S1 and S2 separately and show their sum. Source activities and the muon flux convert the per-primary simulation into predicted pulse frequencies.

The primary calculation uses raw measured PMT charge and height features. No FFT or Savitzky--Golay filter is applied. The notebook embeds its figures and tables and does not automatically write PNG or CSV files.


In [ ]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import SRC.KingCRAB.radioactive as radioactive_helpers
from SRC.KingCRAB.context import configure_module
from SRC.KingCRAB.radioactive import decode, gamma_cluster_charge, load_digitized, log_extrap, log_interp, optical_rate, plot_configuration, rate_weighted_population, read_source_h5, simulate_optical_population

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from urllib.parse import unquote
from scipy.stats import binom
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)
ROOT=PROJECT_ROOT
if ROOT.name=='CODE': ROOT=ROOT.parent
DATA=ROOT/'DATA'
NEXUS=Path('/Users/physics/packages/nexus')

BIN_NS=10.0
MUON_RATE_HZ=49.217
BA_ACTIVITY_UCI=0.34
EU_ACTIVITY_UCI=0.44
UCI_TO_BQ=3.7e4
PRIMARY_RATES_HZ={'Muon':MUON_RATE_HZ,
                  'Ba-133':BA_ACTIVITY_UCI*UCI_TO_BQ,
                  'Eu-152':EU_ACTIVITY_UCI*UCI_TO_BQ}
SEED=410000
SPE_RELATIVE_SIGMA=0.35
E_CHARGE_C=1.602176634e-19
PMT_VOLTAGE_V=900.0

configure_module(radioactive_helpers, globals())



spectral=load_digitized(DATA/'R7378A_Spectral_Response.json')
characteristics=load_digitized(DATA/'R7378A_Characteristics.json')
qe=log_extrap(spectral['Quantum Efficiency'],128.0)/100
radiant=log_extrap(spectral['Cathode Radiant Sensitivity'],128.0)
qe_from_radiant=124.0*radiant/128.0/100
PDE=float(np.sqrt(qe*qe_from_radiant))
gain_points=characteristics['Gain'].copy(); gain_points[:,1]*=1e13
GAIN=log_interp(gain_points,PMT_VOLTAGE_V)
Q_SPE_C=GAIN*E_CHARGE_C

print(f'PDE at 128 nm: {100*PDE:.3f}%')
print(f'Gain at 900 V: {GAIN:,.0f}; mean SPE charge: {Q_SPE_C*1e12:.5f} pC')
print(f'Ba-133 activity: {PRIMARY_RATES_HZ["Ba-133"]:,.0f} decays/s')
print(f'Eu-152 activity: {PRIMARY_RATES_HZ["Eu-152"]:,.0f} decays/s')

## 1. Measured PMT background and charge-to-height response

The terminal-4 dark acquisition supplies the real background population. Its timestamp-derived live time defines the selected-pulse frequency. Selected records provide the empirical joint relationship between integrated charge and negative pulse height, including PMT gain variation, electronics response, and non-Gaussian tails.

For an injected $k$-PE optical cluster, the charge is drawn from a gamma SPE model with mean $kQ_{\rm SPE}$ and 35% single-PE relative width. Its height is obtained by resampling the measured height-to-charge ratio from real positive-charge selected pulses. This predicts population distributions without pretending that a dark-current waveform is an illuminated SPE calibration.


## 2. Nexus photon arrivals

The muon tables are read from `DATA/cosmogenics`. Ba-133 and Eu-152 are read directly from the four local Nexus HDF5 files. Only unique optical photons crossing `II_ONE_INCH_SCORE` are retained. `ORIGIN:Scintillation` is S1 and `ORIGIN:Electroluminescence` is S2.

The embedded HDF5 configuration is checked so a mislabeled isotope, field, gas, or detector plane fails visibly rather than silently entering the rate model.


In [ ]:
mu=pd.read_csv(DATA/'cosmogenics/photon_arrivals.csv')
mu=mu[['condition','event_id','particle_id','origin','arrival_time_ns']].copy()
mu['source']='Muon'
mu_events=pd.read_csv(DATA/'cosmogenics/paired_event_summary.csv').event_id.nunique()

H5={('Ba-133','field_off'):NEXUS/'KingCRAB_BaGen_FieldOff.h5',
    ('Ba-133','field_on'): NEXUS/'KingCRAB_BaGen_FieldOn.h5',
    ('Eu-152','field_off'):NEXUS/'KingCRAB_EuGen_FieldOff.h5',
    ('Eu-152','field_on'): NEXUS/'KingCRAB_EuGen_FieldOn.h5'}

configure_module(radioactive_helpers, globals())


frames=[mu]; event_counts={'Muon':mu_events}; validation=[]
for (source,condition),path in H5.items():
    frame,n,checks=read_source_h5(source,condition,path); frames.append(frame)
    event_counts[source]=n
    validation.append({'source':source,'condition':condition,'events':n,
                       'photons':len(frame),**checks})
photons=pd.concat(frames,ignore_index=True)
display(pd.DataFrame(validation))
display(photons.groupby(['source','condition','origin']).size().rename('photons').to_frame())

## 3. Fixed 10 ns multiplicity and physical frequency

For primary $i$ and time bin $b$,

\[
n_{ib}=\#\{\text{score-plane photons in }[10b,10(b+1))\ \mathrm{ns}\}.
\]

The detected PE multiplicity is $K_{ib}\sim\mathrm{Binomial}(n_{ib},p_{\rm PDE})$. Empty bins are irrelevant to pulse triggering. A simulated bin contributes a physical frequency equal to its occurrence per generated primary multiplied by the primary rate. This preserves both photon multiplicity and the simulated arrival-time structure.


In [ ]:
photons['time_bin_10ns']=np.floor(photons.arrival_time_ns/BIN_NS).astype(np.int64)
bins=(photons.groupby(['source','condition','origin','event_id','time_bin_10ns'])
      .size().rename('incident_photons').reset_index())

frequency=[]
for (source,condition,origin),g in bins.groupby(['source','condition','origin']):
    nprim=event_counts[source]; primary_rate=PRIMARY_RATES_HZ[source]
    mult,freq=np.unique(g.incident_photons,return_counts=True)
    for n,f in zip(mult,freq):
        frequency.append({'source':source,'condition':condition,'origin':origin,
                          'incident_photons_in_10ns':int(n),'observed_bins':int(f),
                          'bins_per_primary':f/nprim,
                          'physical_bin_frequency_Hz':primary_rate*f/nprim,
                          'P_at_least_1_PE':binom.sf(0,n,PDE),
                          'P_at_least_2_PE':binom.sf(1,n,PDE)})
frequency_table=pd.DataFrame(frequency)

rate_summary=[]
for keys,g in frequency_table.groupby(['source','condition','origin']):
    source,condition,origin=keys
    rate_summary.append({'source':source,'condition':condition,'origin':origin,
        'detected_cluster_rate_Hz':np.sum(g.physical_bin_frequency_Hz*g.P_at_least_1_PE),
        'multi_PE_rate_Hz':np.sum(g.physical_bin_frequency_Hz*g.P_at_least_2_PE)})
rate_summary=pd.DataFrame(rate_summary)
display(rate_summary.pivot_table(index='source',columns=['condition','origin'],
                                 values=['detected_cluster_rate_Hz','multi_PE_rate_Hz'],fill_value=0).round(4))

## 4. Simulated charge and height distributions

The Monte Carlo below samples 10 ns photon bins in proportion to their physical frequency, then samples the detected PE count. Only bins producing at least one PE enter the optical-pulse distribution. S1 and S2 labels remain separate. The measured field-off background population is shown independently and is also mixed with each optical component according to rate for the combined prediction.

These are distributions of triggered pulses, not random 10 ns windows. Their integrals are therefore normalized as probability densities; the accompanying rate table supplies the absolute frequency.


In [ ]:
configure_module(radioactive_helpers, globals())


pop=[]
for si,source in enumerate(PRIMARY_RATES_HZ):
    for ci,condition in enumerate(['field_off','field_on']):
        origins=['S1'] if condition=='field_off' else ['S1','S2']
        for oi,origin in enumerate(origins):
            pop.append(simulate_optical_population(source,condition,origin,
                                                   seed=SEED+100*si+10*ci+oi))
optical=pd.concat(pop,ignore_index=True)

background=dark_selected[['height_V','charge_C']].copy()
background['source']='Measured background'; background['condition']='field_off'
background['origin']='background'; background['detected_pe']=background.charge_C/Q_SPE_C

field_off_opt=optical.query("condition == 'field_off' and origin == 'S1'")
field_on_opt=optical.query("condition == 'field_on' and origin in ['S1','S2']")

q_all=np.r_[background.charge_C,field_off_opt.charge_C,field_on_opt.charge_C]*1e12
h_all=np.r_[background.height_V,field_off_opt.height_V,field_on_opt.height_V]*1e3
qmax=np.quantile(q_all,.995); hmax=np.quantile(h_all,.995)



# Muons are included in every source-added curve.  The background-only curve
# is the measured PMT population and deliberately contains no simulated muons.
off_specs=[('Background',[],'k'),
           ('Field Off (Ba)',['Muon','Ba-133'],'tab:red'),
           ('Field Off (Eu)',['Muon','Eu-152'],'tab:cyan'),
           ('Field Off (Both)',['Muon','Ba-133','Eu-152'],'tab:orange')]
on_specs=[('Background + simulated muons',['Muon'],'tab:purple'),
          ('Field On (Ba)',['Muon','Ba-133'],'tab:red'),
          ('Field On (Eu)',['Muon','Eu-152'],'tab:cyan'),
          ('Field On (Both)',['Muon','Ba-133','Eu-152'],'tab:blue')]

off_populations=[]; on_populations=[]
for i,(label,sources,color) in enumerate(off_specs):
    frame,rate=rate_weighted_population('field_off',sources,seed=SEED+9100+i)
    off_populations.append((frame,label,color,rate))
for i,(label,sources,color) in enumerate(on_specs):
    frame,rate=rate_weighted_population('field_on',sources,seed=SEED+9200+i)
    on_populations.append((frame,label,color,rate))


plot_configuration(off_populations,'Field off: measured background and S1 populations')
plot_configuration(on_populations,'Field on: measured background, muons, and source S1 + S2')

dist_summary=pd.concat([background.assign(population='Measured background'),optical],ignore_index=True).groupby('population').agg(
    pulses=('charge_C','size'),median_PE=('detected_pe','median'),
    p90_PE=('detected_pe',lambda x:x.quantile(.9)),
    median_charge_pC=('charge_C',lambda x:1e12*x.median()),
    p90_charge_pC=('charge_C',lambda x:1e12*x.quantile(.9)),
    median_height_mV=('height_V',lambda x:1e3*x.median()),
    p90_height_mV=('height_V',lambda x:1e3*x.quantile(.9)))
display(dist_summary.round(5))

## 5. Combined field-off and field-on rates

The aggregate prediction adds rates, not unweighted histograms. Field off contains measured background plus S1 from muons and both sources. Field on contains the same measured background and S1 populations plus the simulated S2 populations. This assumes the sources remain in CRAB simultaneously and that their stated activities are current activities at acquisition time.


In [ ]:
components=rate_summary.copy()
components['component']=components.source+' '+components.origin
off=components.query("condition=='field_off' and origin=='S1'")
on=components.query("condition=='field_on' and origin in ['S1','S2']")

totals=pd.DataFrame([
 {'configuration':'field off: background + all S1',
  'detected_cluster_rate_Hz':BACKGROUND_RATE_HZ+off.detected_cluster_rate_Hz.sum(),
  'multi_PE_rate_Hz':BACKGROUND_RATE_HZ*np.mean(background.charge_C>=1.5*Q_SPE_C)+off.multi_PE_rate_Hz.sum()},
 {'configuration':'field on: background + all S1 + all S2',
  'detected_cluster_rate_Hz':BACKGROUND_RATE_HZ+on.detected_cluster_rate_Hz.sum(),
  'multi_PE_rate_Hz':BACKGROUND_RATE_HZ*np.mean(background.charge_C>=1.5*Q_SPE_C)+on.multi_PE_rate_Hz.sum()}])
display(pd.concat([pd.DataFrame([{'source':'Measured background','condition':'both','origin':'background',
   'detected_cluster_rate_Hz':BACKGROUND_RATE_HZ,
   'multi_PE_rate_Hz':BACKGROUND_RATE_HZ*np.mean(background.charge_C>=1.5*Q_SPE_C)}]),components],ignore_index=True).round(5))
display(totals.round(5))

labels=[]; off_rates=[]; on_rates=[]
for source in PRIMARY_RATES_HZ:
    labels.append(source)
    off_rates.append(off.query('source==@source').detected_cluster_rate_Hz.sum())
    on_rates.append(on.query('source==@source').detected_cluster_rate_Hz.sum())
labels=['Measured background']+labels
off_rates=[BACKGROUND_RATE_HZ]+off_rates; on_rates=[BACKGROUND_RATE_HZ]+on_rates
x=np.arange(len(labels)); width=.38
fig,ax=plt.subplots(figsize=(10,5))
ax.bar(x-width/2,off_rates,width,label='field off: S1',color='tab:orange')
ax.bar(x+width/2,on_rates,width,label='field on: S1 + S2',color='tab:blue')
ax.set_yscale('log'); ax.set_xticks(x,labels); ax.set_ylabel('predicted detected 10 ns cluster rate [Hz]')
ax.set_title('Absolute pulse frequency with both sources in CRAB'); ax.legend(); fig.tight_layout(); plt.show()

## Interpretation and limitations

- A 10 ns bin is the declared resolving interval. Changing it changes the multi-PE rate and must be treated as an acquisition/reconstruction systematic.
- Source rates scale linearly with activity. The calculation assumes every stated decay is represented by the simulated isotope decay and that source encapsulation and placement are already represented by Nexus.
- The detector-score photons are converted with the extrapolated R7378A PDE. The VUV PDE is a leading normalization uncertainty.
- Charge and height are correlated using measured terminal-4 PMT pulses. This is more defensible than hard-coding a voltage gain, but a dedicated low-light SPE calibration would improve the response model.
- The field-on HDF5 files were generated with the prior KingCRAB field implementation. They should be regenerated after the clean NEXT-100-style `ACTIVE`/`EL_GAP` handoff before quoting the final prediction.
